# Imbalanced Numerical Data: Oversampling and Undersampling
# 数値データの不均衡対策：オーバーサンプリングとアンダーサンプリング

## 日本語
このノートブックは、数値データを使った二値分類におけるクラス不均衡を、初学者向けに説明します。多数クラスを `0`、少数クラスを `1` とします。

主な内容：
- 不均衡データとは何か
- Random Oversampling
- SMOTE
- Random Undersampling
- 正しいデータ分割とデータリーク対策
- 交差検証による公平な比較
- Precision、Recall、F1、PR-AUCなどによる評価

## English
This notebook explains class imbalance in binary classification with numerical features. It assumes that class `0` is the majority class and class `1` is the minority class.

Main topics:
- What imbalanced data means
- Random Oversampling
- SMOTE
- Random Undersampling
- Correct data splitting and leakage prevention
- Fair comparison with cross-validation
- Evaluation with precision, recall, F1, PR-AUC, and related metrics


## 1. What Is Imbalanced Data? / 不均衡データとは

### 日本語
不均衡データとは、一方のクラスがもう一方より非常に多いデータです。たとえば、正常データが9,500件、異常データが500件の場合、異常は全体の5%です。

すべてを「正常」と予測してもAccuracyは95%になります。しかし、異常を1件も検出できないため、実用的なモデルではありません。そのため、Accuracyだけではなく、少数クラスのRecall、Precision、F1、PR-AUCを確認します。

### English
Imbalanced data occurs when one class has many more observations than another. For example, a dataset with 9,500 normal cases and 500 abnormal cases has a 5% minority rate.

A model that predicts every case as normal obtains 95% accuracy but detects no abnormal cases. Therefore, accuracy alone is misleading. Minority-class recall, precision, F1, and PR-AUC should also be evaluated.


## 2. Method Comparison / 方法の比較

| Method / 方法 | 日本語の説明 | English explanation | Main risk / 主な注意点 |
|---|---|---|---|
| Random Oversampling | 少数クラスをランダムに複製する | Randomly duplicates minority observations | 重複による過学習 / Overfitting from duplicates |
| SMOTE | 近傍の少数クラス間を補間して合成データを作る | Interpolates synthetic minority observations | ノイズやクラス重複を増幅する可能性 / May amplify noise or overlap |
| Random Undersampling | 多数クラスをランダムに削減する | Randomly removes majority observations | 重要な情報を失う可能性 / May discard useful information |

### 日本語の選び方
- データが小さい：まずRandom Oversampling、次にSMOTEを比較
- 数値特徴が連続値：SMOTEが候補
- 多数クラスが非常に大きい：Random Undersamplingが候補
- どの方法でも、未処理のベースラインと同じ交差検証条件で比較

### English selection guidance
- Small dataset: compare Random Oversampling first, then SMOTE
- Continuous numerical features: SMOTE is a reasonable candidate
- Extremely large majority class: Random Undersampling may reduce computation
- Always compare sampling methods with an untreated baseline under identical cross-validation folds


## 3. Important Rules / 重要なルール

| Rule | 日本語 | English |
|---|---|---|
| Split first | サンプリング前に訓練・検証・テストへ分割する | Split before resampling |
| Training only | サンプリングは訓練データだけに適用する | Apply sampling only to training data |
| Pipeline | 交差検証では `imblearn.pipeline.Pipeline` を使う | Use `imblearn.pipeline.Pipeline` in cross-validation |
| Scaling | SMOTEの前に数値特徴を標準化する | Scale numerical features before SMOTE |
| Test set | テストデータは最後まで未使用にする | Keep the test set untouched until final evaluation |
| Primary metric | 少数クラスが希少ならPR-AUCを重視する | Prefer PR-AUC when the positive class is rare |
| Sampling ratio | 最初から1:1に固定せず0.25、0.5、0.75、1.0を比較する | Compare ratios such as 0.25, 0.5, 0.75, and 1.0 |


## 4. Install and Import Libraries / ライブラリの準備

### 日本語
次のセルはGoogle Colabで必要なライブラリをインストールし、分析に使用する機能を読み込みます。

### English
The next cells install the required package in Google Colab and import the tools used in the analysis.


In [ ]:
# Install the package that provides oversampling and undersampling methods.
!pip install -q imbalanced-learn


In [ ]:
# Import numerical and tabular data libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import data generation, splitting, preprocessing, and modeling tools.
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics and visual tools.
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score
)

# Import leakage-safe pipelines and sampling methods.
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 5. Create Example Numerical Data / 数値サンプルデータの作成

### 日本語
10,000件、数値特徴20個の二値分類データを作成します。少数クラスは約5%です。実データを使う場合は、後半のCSV読み込みセルに置き換えてください。

### English
This section creates 10,000 observations with 20 numerical features. Approximately 5% belong to the minority class. Replace this section with the CSV-loading cell later in the notebook when using real data.


In [ ]:
# Create a reproducible imbalanced numerical dataset.
X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    weights=[0.95, 0.05],
    class_sep=1.0,
    flip_y=0.01,
    random_state=42
)

# Convert arrays to pandas objects with readable names.
X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(1, 21)])
y = pd.Series(y, name="target")

class_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Proportion": y.value_counts(normalize=True).sort_index()
})
print("Original class distribution:")
display(class_distribution)


## 6. Stratified Train, Validation, and Test Split / 層化分割

### 日本語
クラス比率を保つため `stratify` を使います。訓練データは学習と交差検証、検証データは閾値選択、テストデータは最終評価だけに使用します。

### English
`stratify` preserves the class ratio. Training data is used for fitting and cross-validation, validation data for threshold selection, and test data only for final evaluation.


In [ ]:
# Reserve 20% of the full dataset as an untouched test set.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Reserve 20% of the remaining data as a validation set.
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=42
)

split_counts = pd.DataFrame({
    "Training": y_train.value_counts().sort_index(),
    "Validation": y_validation.value_counts().sort_index(),
    "Test": y_test.value_counts().sort_index()
})
print("Class counts by split:")
display(split_counts)


## 7. Visualize the Imbalance / 不均衡の可視化

### 日本語
訓練データのクラス数を棒グラフで確認します。

### English
The bar chart shows the class counts in the training data.


In [ ]:
# Plot the training class distribution.
y_train.value_counts().sort_index().plot(kind="bar")
plt.title("Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Random Oversampling / ランダム・オーバーサンプリング

### 日本語
少数クラスの既存データをランダムに複製します。簡単ですが、同じデータが繰り返されるため過学習の可能性があります。

`sampling_strategy=0.5` は、処理後の少数クラス数を多数クラス数の50%にする設定です。

### English
Random Oversampling duplicates existing minority observations. It is simple, but repeated observations may increase overfitting.

`sampling_strategy=0.5` makes the minority count 50% of the majority count after resampling.


In [ ]:
# Demonstrate Random Oversampling on the training data only.
random_over_sampler = RandomOverSampler(
    sampling_strategy=0.5,
    random_state=42
)
X_random_over, y_random_over = random_over_sampler.fit_resample(X_train, y_train)

random_over_counts = pd.DataFrame({
    "Before": y_train.value_counts().sort_index(),
    "After": y_random_over.value_counts().sort_index()
})
print("Random Oversampling class counts:")
display(random_over_counts)


## 9. SMOTE / 合成オーバーサンプリング

### 日本語
SMOTEは、近い少数クラスの点同士を補間して新しい点を作ります。連続値の数値特徴に適しています。ただし、外れ値やクラス境界の重なりが多い場合、好ましくない合成点を作ることがあります。

### English
SMOTE creates synthetic observations between neighboring minority samples. It is suitable for continuous numerical features. However, it may create undesirable synthetic points when minority outliers or strong class overlap are present.


In [ ]:
# Scale numerical features before the standalone SMOTE demonstration.
standalone_scaler = StandardScaler()
X_train_scaled = standalone_scaler.fit_transform(X_train)

# Demonstrate SMOTE on the scaled training data only.
smote_sampler = SMOTE(
    sampling_strategy=0.5,
    k_neighbors=5,
    random_state=42
)
X_smote, y_smote = smote_sampler.fit_resample(X_train_scaled, y_train)

smote_counts = pd.DataFrame({
    "Before": y_train.value_counts().sort_index(),
    "After": y_smote.value_counts().sort_index()
})
print("SMOTE class counts:")
display(smote_counts)


## 10. Random Undersampling / ランダム・アンダーサンプリング

### 日本語
多数クラスをランダムに削減します。学習時間を短縮できますが、重要な多数クラスの情報を失う可能性があります。多数クラスが非常に大きい場合に有効です。

### English
Random Undersampling removes majority observations. It reduces training time but may discard useful majority information. It is most attractive when the majority class is extremely large.


In [ ]:
# Demonstrate Random Undersampling on the training data only.
random_under_sampler = RandomUnderSampler(
    sampling_strategy=0.5,
    random_state=42
)
X_random_under, y_random_under = random_under_sampler.fit_resample(X_train, y_train)

random_under_counts = pd.DataFrame({
    "Before": y_train.value_counts().sort_index(),
    "After": y_random_under.value_counts().sort_index()
})
print("Random Undersampling class counts:")
display(random_under_counts)


## 11. Leakage-Safe Pipelines / データリークを防ぐPipeline

### 日本語
交差検証の前にデータ全体へSMOTEを適用すると、検証foldの情報が学習foldへ混入します。`imblearn.pipeline.Pipeline`を使うと、各foldの訓練部分だけで標準化とサンプリングが実行されます。

### English
Applying SMOTE before cross-validation leaks information from validation folds into training folds. `imblearn.pipeline.Pipeline` ensures that scaling and sampling are fitted only within each training fold.


In [ ]:
# Define comparable leakage-safe workflows.
methods = {
    "No Sampling": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Random Oversampling": Pipeline([
        ("scaler", StandardScaler()),
        ("sampler", RandomOverSampler(sampling_strategy=0.5, random_state=42)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "SMOTE": Pipeline([
        ("scaler", StandardScaler()),
        ("sampler", SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=42)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Random Undersampling": Pipeline([
        ("scaler", StandardScaler()),
        ("sampler", RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ])
}


## 12. Cross-Validation Comparison / 交差検証による比較

### 日本語
同じ5-fold層化交差検証で4方法を比較します。主指標はPR-AUCです。Recallは見逃しの少なさ、Precisionは陽性予測の正確さ、F1は両者のバランスを表します。

### English
The four methods are compared using the same five-fold stratified cross-validation. PR-AUC is the primary metric. Recall measures detection coverage, precision measures the reliability of positive predictions, and F1 balances both.


In [ ]:
# Define reproducible stratified cross-validation.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define evaluation metrics.
scoring = {
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "PR_AUC": "average_precision",
    "ROC_AUC": "roc_auc",
    "Balanced_Accuracy": "balanced_accuracy"
}

comparison_rows = []
for method_name, pipeline in methods.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    comparison_rows.append({
        "Method": method_name,
        "Precision": scores["test_Precision"].mean(),
        "Recall": scores["test_Recall"].mean(),
        "F1": scores["test_F1"].mean(),
        "PR-AUC": scores["test_PR_AUC"].mean(),
        "ROC-AUC": scores["test_ROC_AUC"].mean(),
        "Balanced Accuracy": scores["test_Balanced_Accuracy"].mean()
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "PR-AUC", ascending=False
).reset_index(drop=True)

print("Cross-validation comparison:")
display(comparison_df.round(4))


## 13. Validation-Based Threshold Selection / 検証データによる閾値選択

### 日本語
分類閾値0.5が最適とは限りません。ここでは交差検証でPR-AUCが最も高い方法を選び、検証データでF1が最大になる閾値を決めます。テストデータで閾値を決めてはいけません。

### English
A classification threshold of 0.5 is not always optimal. This section selects the pipeline with the highest cross-validation PR-AUC and uses validation data to find the threshold that maximizes F1. The test set must not be used for threshold selection.


In [ ]:
# Select the pipeline with the highest cross-validation PR-AUC.
best_method_name = comparison_df.loc[0, "Method"]
best_pipeline = methods[best_method_name]
best_pipeline.fit(X_train, y_train)
print(f"Selected method: {best_method_name}")

# Predict validation probabilities and find the F1-maximizing threshold.
validation_probability = best_pipeline.predict_proba(X_validation)[:, 1]
precision_values, recall_values, thresholds = precision_recall_curve(
    y_validation, validation_probability
)
f1_values = (
    2 * precision_values[:-1] * recall_values[:-1]
    / (precision_values[:-1] + recall_values[:-1] + 1e-10)
)
best_index = np.argmax(f1_values)
best_threshold = float(thresholds[best_index])

print(f"Validation-selected threshold: {best_threshold:.4f}")
print(f"Validation F1-score: {f1_values[best_index]:.4f}")


## 14. Final Test Evaluation / 最終テスト評価

### 日本語
未使用のテストデータで、既定閾値0.5と検証データで選んだ閾値を比較します。

### English
The untouched test set is used to compare the default 0.5 threshold with the threshold selected on validation data.


In [ ]:
# Generate test probabilities and predictions.
test_probability = best_pipeline.predict_proba(X_test)[:, 1]
test_prediction_default = (test_probability >= 0.50).astype(int)
test_prediction_selected = (test_probability >= best_threshold).astype(int)

def metric_row(label, threshold, prediction):
    """Return a consistent set of classification metrics."""
    return {
        "Threshold Type": label,
        "Threshold": threshold,
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction, zero_division=0),
        "PR-AUC": average_precision_score(y_test, test_probability),
        "ROC-AUC": roc_auc_score(y_test, test_probability),
        "Balanced Accuracy": balanced_accuracy_score(y_test, prediction),
        "MCC": matthews_corrcoef(y_test, prediction)
    }

final_results = pd.DataFrame([
    metric_row("Default", 0.50, test_prediction_default),
    metric_row("Validation-selected", best_threshold, test_prediction_selected)
])
print("Final test results:")
display(final_results.round(4))

print("Classification report for the validation-selected threshold:")
print(classification_report(y_test, test_prediction_selected, digits=4))


## 15. Confusion Matrices and Precision-Recall Curve / 混同行列とPR曲線

### 日本語
混同行列で偽陽性と偽陰性を確認します。PR曲線は、少数クラスが希少な場合のPrecisionとRecallのトレードオフを示します。

### English
Confusion matrices show false positives and false negatives. The precision-recall curve shows the trade-off between precision and recall when the positive class is rare.


In [ ]:
# Compare confusion matrices for the two thresholds.
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction_default,
    display_labels=["Majority (0)", "Minority (1)"],
    colorbar=False, ax=axes[0]
)
axes[0].set_title("Default Threshold = 0.50")

ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction_selected,
    display_labels=["Majority (0)", "Minority (1)"],
    colorbar=False, ax=axes[1]
)
axes[1].set_title(f"Validation Threshold = {best_threshold:.3f}")
plt.tight_layout()
plt.show()

# Plot the test-set precision-recall curve.
test_precision, test_recall, test_thresholds = precision_recall_curve(
    y_test, test_probability
)
closest_index = np.argmin(np.abs(test_thresholds - best_threshold))
plt.figure(figsize=(7, 5))
plt.plot(test_recall, test_precision,
         label=f"PR-AUC = {average_precision_score(y_test, test_probability):.3f}")
plt.scatter(test_recall[closest_index], test_precision[closest_index],
            label=f"Selected threshold = {best_threshold:.3f}")
plt.title("Test-Set Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.show()


## 16. Compare Sampling Ratios / サンプリング比率の比較

### 日本語
1:1に揃えることが常に最適とは限りません。SMOTEの比率0.25、0.5、0.75、1.0を同じ交差検証で比較します。

### English
A 1:1 class balance is not always optimal. This section compares SMOTE ratios of 0.25, 0.5, 0.75, and 1.0 under identical cross-validation.


In [ ]:
# Compare several SMOTE minority-to-majority ratios.
ratio_rows = []
for ratio in [0.25, 0.50, 0.75, 1.00]:
    ratio_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("sampler", SMOTE(sampling_strategy=ratio, k_neighbors=5, random_state=42)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ])
    scores = cross_validate(
        ratio_pipeline, X_train, y_train, cv=cv,
        scoring={"PR_AUC": "average_precision", "Precision": "precision",
                 "Recall": "recall", "F1": "f1"},
        n_jobs=-1
    )
    ratio_rows.append({
        "Sampling Ratio": ratio,
        "PR-AUC": scores["test_PR_AUC"].mean(),
        "Precision": scores["test_Precision"].mean(),
        "Recall": scores["test_Recall"].mean(),
        "F1": scores["test_F1"].mean()
    })

ratio_results = pd.DataFrame(ratio_rows).sort_values("PR-AUC", ascending=False)
print("SMOTE sampling-ratio comparison:")
display(ratio_results.round(4))


## 17. Use Your Own CSV File / 独自CSVの使用

### 日本語
以下のセルを有効化し、ファイルパスと目的変数名を変更してください。このノートブックは数値特徴のみを対象にします。欠損値がある場合は、モデル学習前に補完または除外が必要です。

### English
Uncomment the following cell and change the file path and target-column name. This notebook uses numerical predictors only. Missing values must be imputed or removed before model fitting.


In [ ]:
# Uncomment and edit this section to use your own CSV file.

# file_path = "/content/your_dataset.csv"
# target_column = "target"
#
# data = pd.read_csv(file_path)
# X = data.drop(columns=[target_column]).select_dtypes(include=["number"]).copy()
# y = data[target_column].copy()
#
# print("Target distribution:")
# display(y.value_counts(dropna=False))
#
# print("Missing numerical values:")
# display(X.isna().sum().sort_values(ascending=False).head(20))
#
# if y.nunique(dropna=True) != 2:
#     raise ValueError("This notebook requires a binary target with exactly two classes.")


## 18. Final Practical Recommendation / 実務的な結論

### 日本語
1. `stratify`で訓練・検証・テストへ分割する  
2. サンプリングなしのモデルを基準にする  
3. Random Oversampling、SMOTE、Random Undersamplingを同じfoldで比較する  
4. 主指標としてPR-AUCを使い、Recall、Precision、F1も確認する  
5. `Pipeline`内で標準化とサンプリングを行う  
6. 閾値は検証データで選ぶ  
7. 最後に未使用のテストデータで1回だけ評価する  

### English
1. Create stratified training, validation, and test sets.  
2. Establish a no-sampling baseline.  
3. Compare Random Oversampling, SMOTE, and Random Undersampling under identical folds.  
4. Use PR-AUC as the primary metric and inspect recall, precision, and F1.  
5. Perform scaling and sampling inside a pipeline.  
6. Select the probability threshold on validation data.  
7. Evaluate once on the untouched test set.
